# Face Recognition using CNN — LFW Dataset

| | |
|---|---|
| **Name** | Ayush Ranjan |
| **Roll No** | 23MIP10135 |
| **Institute** | VIT Bhopal |
| **Dataset** | [Labeled Faces in the Wild (LFW)](https://www.kaggle.com/datasets/jessicali9530/lfw-dataset) — Kaggle |

**Objective:** Build a CNN that recognises individuals from face images. 
The LFW dataset contains face images of celebrities/public figures collected from the web. 
We select the top individuals with the most images (≥ 20 per person) for multi-class classification.


## Step 1: Install Libraries


In [ ]:
!pip install kaggle tensorflow scikit-learn matplotlib seaborn pillow -q
print('Libraries ready!')

## Step 2: Kaggle API Setup
> Run this cell — it configures Kaggle credentials automatically.


In [ ]:
import os

# ── Kaggle API Token (new KGAT format — no username required) ──
os.environ['KAGGLE_KEY'] = 'KGAT_a8751a0f54996277f33924e203785e29'

# Write kaggle.json for CLI compatibility
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    import json
    json.dump({"key": "KGAT_a8751a0f54996277f33924e203785e29"}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API configured!')

## Step 3: Download LFW Dataset


In [ ]:
!kaggle datasets download -d jessicali9530/lfw-dataset -p /content/lfw_raw --unzip -q
print('Downloaded!')
!ls /content/lfw_raw/

## Step 4: Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, shutil, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Dense, Dropout,
                                      BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)
tf.random.set_seed(42)
print(f'TensorFlow {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

## Step 5: Explore & Filter Dataset

> LFW has 5,749 individuals but most have only 1–2 images. 
> We keep individuals with **≥ 20 images** to ensure enough data per class.


In [ ]:
# Locate the image directory (structure varies by kaggle download)
raw_base = '/content/lfw_raw'
for root, dirs, files in os.walk(raw_base):
    if dirs and not any(d.startswith('.') for d in dirs):
        sample = os.listdir(os.path.join(root, dirs[0]))
        if any(f.endswith('.jpg') for f in sample):
            LFW_DIR = root
            break
print(f'LFW image directory: {LFW_DIR}')

# Count images per person
MIN_IMAGES = 20
person_counts = {}
for person in os.listdir(LFW_DIR):
    p = os.path.join(LFW_DIR, person)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.jpg')])
        if n >= MIN_IMAGES:
            person_counts[person] = n

# Sort and take top 15
person_counts = dict(sorted(person_counts.items(), key=lambda x: -x[1])[:15])
print(f'Selected {len(person_counts)} individuals (>= {MIN_IMAGES} images each):')
for p, n in person_counts.items():
    print(f'  {p.replace("_"," ")}: {n} images')
print(f'  TOTAL: {sum(person_counts.values())} images')

In [ ]:
# Bar chart of selected people
plt.figure(figsize=(14, 5))
names = [p.replace('_', ' ') for p in person_counts.keys()]
counts = list(person_counts.values())
bars = plt.bar(names, counts, color=plt.cm.tab20.colors[:len(names)], edgecolor='k', lw=0.7)
for bar, v in zip(bars, counts):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, str(v),
             ha='center', fontsize=9, fontweight='bold')
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.title('LFW Selected Individuals — Image Count', fontsize=14, fontweight='bold')
plt.ylabel('Images'); plt.tight_layout()
plt.savefig('lfw_class_dist.png', dpi=150, bbox_inches='tight'); plt.show()

# Sample faces
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
axes = axes.flatten()
for i, person in enumerate(person_counts.keys()):
    p = os.path.join(LFW_DIR, person)
    img = load_img(os.path.join(p, os.listdir(p)[0]), target_size=(128, 128))
    axes[i].imshow(img)
    axes[i].set_title(person.replace('_',' '), fontsize=9, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('LFW Dataset — Selected Individuals', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('lfw_samples.png', dpi=150, bbox_inches='tight'); plt.show()

## Step 6: Create Train / Validation Split


In [ ]:
import random
random.seed(42)

split_dir = '/content/lfw_split'
VAL_RATIO  = 0.20

if not os.path.exists(split_dir):
    for split in ['train', 'val']:
        for person in person_counts:
            os.makedirs(os.path.join(split_dir, split, person), exist_ok=True)

    for person in person_counts:
        src   = os.path.join(LFW_DIR, person)
        imgs  = [f for f in os.listdir(src) if f.endswith('.jpg')]
        random.shuffle(imgs)
        n_val = max(1, int(len(imgs) * VAL_RATIO))
        for f in imgs[n_val:]:  shutil.copy(os.path.join(src, f), os.path.join(split_dir, 'train', person, f))
        for f in imgs[:n_val]:  shutil.copy(os.path.join(src, f), os.path.join(split_dir, 'val',   person, f))

    print('Split created!')
else:
    print('Split already exists.')

for s in ['train', 'val']:
    tot = sum(len(os.listdir(os.path.join(split_dir, s, p))) for p in person_counts)
    print(f'{s}: {tot} images')

## Step 7: Data Augmentation & Generators


In [ ]:
IMG_SIZE   = 128
BATCH_SIZE = 16
EPOCHS     = 60

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.25,
    shear_range=0.2,
    brightness_range=[0.75, 1.25],
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(split_dir, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=42
)
val_gen = val_datagen.flow_from_directory(
    os.path.join(split_dir, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

NUM_CLASSES = len(train_gen.class_indices)
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'Classes: {NUM_CLASSES} | Train batches: {len(train_gen)} | Val batches: {len(val_gen)}')

## Step 8: Build Deep CNN


In [ ]:
def build_face_cnn(num_classes, input_shape=(128, 128, 3)):
    """Deep CNN for LFW Face Recognition — 5 conv blocks + GAP + Dense head"""
    model = Sequential([
        # Block 1 — 32 filters
        Conv2D(32,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4),input_shape=input_shape),
        BatchNormalization(), Conv2D(32,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling2D((2,2)), Dropout(0.2),
        # Block 2 — 64 filters
        Conv2D(64,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Conv2D(64,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling2D((2,2)), Dropout(0.25),
        # Block 3 — 128 filters
        Conv2D(128,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Conv2D(128,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling2D((2,2)), Dropout(0.3),
        # Block 4 — 256 filters
        Conv2D(256,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Conv2D(256,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling2D((2,2)), Dropout(0.3),
        # Block 5 — 512 filters
        Conv2D(512,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.3),
        # Classifier
        GlobalAveragePooling2D(),
        Dense(512,activation='relu',kernel_regularizer=l2(1e-4)), BatchNormalization(), Dropout(0.5),
        Dense(256,activation='relu',kernel_regularizer=l2(1e-4)), Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ], name='FaceRecognition_CNN')
    return model

model = build_face_cnn(NUM_CLASSES)
model.summary()
print(f'Total params: {model.count_params():,}')

## Step 9: Compile & Train


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_acc')]
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=1),
    ModelCheckpoint('/content/best_face_cnn.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

history = model.fit(
    train_gen, epochs=EPOCHS, validation_data=val_gen, callbacks=callbacks, verbose=1
)
print(f'Best Val Accuracy: {max(history.history["val_accuracy"])*100:.2f}%')

## Step 10: Results & Evaluation


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, metric, label in zip(axes, ['accuracy','loss'], ['Accuracy','Loss']):
    ax.plot(history.history[metric],     'b-o', label='Train', lw=2, ms=4)
    ax.plot(history.history[f'val_{metric}'], 'r-o', label='Validation', lw=2, ms=4)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(label)
    ax.legend(); ax.grid(True, alpha=0.3)
if 'val_accuracy' in history.history:
    best_e = history.history['val_accuracy'].index(max(history.history['val_accuracy']))
    axes[0].axvline(x=best_e, color='g', linestyle='--', alpha=0.7,
                   label=f'Best: {max(history.history["val_accuracy"])*100:.1f}%')
    axes[0].legend()
plt.suptitle('Face Recognition CNN — Training History', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('face_history.png', dpi=150, bbox_inches='tight'); plt.show()

# Evaluate
val_gen.reset()
res = model.evaluate(val_gen, verbose=1)
print(f'Validation Accuracy: {res[1]*100:.2f}%  |  Top-2: {res[2]*100:.2f}%  |  Loss: {res[0]:.4f}')

In [ ]:
# Confusion matrix
val_gen.reset()
y_pred = np.argmax(model.predict(val_gen, verbose=0), axis=1)
y_true = val_gen.classes
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, cmap='Blues'); plt.colorbar(im)
labels = [c.replace('_',' ') for c in CLASS_NAMES]
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(labels, fontsize=8)
thresh = cm.max()/2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=8,fontweight='bold',
                color='white' if cm[i,j]>thresh else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Face Recognition CNN — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('face_confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Sample predictions
val_gen.reset()
imgs, labels = next(val_gen)
preds = model.predict(imgs, verbose=0)
n = min(12, len(imgs))
fig, axes = plt.subplots(3, 4, figsize=(16, 12)); axes = axes.flatten()
for i in range(n):
    ti=np.argmax(labels[i]); pi=np.argmax(preds[i]); conf=preds[i][pi]*100
    axes[i].imshow(imgs[i])
    axes[i].set_title(f'True: {CLASS_NAMES[ti].replace("_"," ")}\nPred: {CLASS_NAMES[pi].replace("_"," ")} ({conf:.1f}%)',
                      color='green' if ti==pi else 'red', fontsize=8, fontweight='bold')
    axes[i].axis('off')
for i in range(n, len(axes)): axes[i].axis('off')
plt.suptitle('Face Recognition — Predictions (Green=Correct, Red=Wrong)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('face_predictions.png', dpi=150, bbox_inches='tight'); plt.show()

## Summary


In [ ]:
print('='*55)
print('   FACE RECOGNITION CNN — SUMMARY')
print('='*55)
print('  Name        : Ayush Ranjan')
print('  Roll No     : 23MIP10135')
print('  Institute   : VIT Bhopal')
print('  Dataset     : Labeled Faces in the Wild (LFW) — Kaggle')
print(f'  Classes     : {NUM_CLASSES} individuals')
print(f'  Image Size  : {IMG_SIZE}x{IMG_SIZE}')
print('  Architecture: Deep CNN — 5 conv blocks + GAP')
print(f'  Parameters  : {model.count_params():,}')
print(f'  Val Accuracy: {res[1]*100:.2f}%')
print(f'  Top-2 Acc   : {res[2]*100:.2f}%')
print('='*55)